# Criação e Otimização do Modelo Preditivo (Machine Learning)Objetivo: Construir um modelo preditivo que classifique a probabilidade do aluno ou aluna entrar em risco de defasagem.Modelos testados previamente: Regressão Logística, Random Forest e XGBoost.Nesta versão otimizada, buscamos a máxima acurácia aplicando: 1. **SMOTE** (balanceamento sintético da classe minoritária)2. **GridSearchCV** (busca dos melhores hiperparâmetros)3. **Inclusão de Features de Variação (Deltas)**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-Learn e ML
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, roc_curve
import pickle


## 1. Carga de Dados

In [ ]:
df = pd.read_csv('dados_limpos_pm.csv')
df.head()


## 2. Feature Engineering & PreparaçãoA variável alvo (target) será `risco_defasagem`. Adicionaremos as variáveis de variação temporal avaliadas no EDA (`delta_ida`, `delta_ieg`, `delta_ips`) pois demonstraram poder de 'alerta precoce'.

In [ ]:
cols_features = ['idade_unificada', 'genero', 'bolsa', 'ida', 'ieg', 'iaa', 'ips', 'ipp', 'ipv', 'delta_ida', 'delta_ieg', 'delta_ips']

X = df[cols_features].copy()
y = df['risco_defasagem'].copy()

# Encode categóricas
le_genero = LabelEncoder()
X['genero'] = le_genero.fit_transform(X['genero'])

le_bolsa = LabelEncoder()
X['bolsa'] = le_bolsa.fit_transform(X['bolsa'])

# Tratamento de Nulos Preenchendo com Mediana (Robusta a Outliers)
X.fillna(X.median(), inplace=True)
X.head()


## 3. Divisão Treino e Teste & Balanceamento com SMOTEO SMOTE criará exemplos sintéticos da classe defasada para o modelo não ficar enviesado para a classe majoritária (Alunos sem Defasagem).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print(f"Treino Inicial: {X_train.shape[0]} | Teste: {X_test.shape[0]}")

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print(f"Treino Pós-SMOTE: {X_train_sm.shape[0]}")


## 4. Otimização do XGBoost com GridSearchCVVamos testar diferentes profundidades de árvore e taxas de aprendizado.

In [ ]:
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)

# Grid de Hyperparâmetros
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [100, 200]
}

# Busca exaustiva cross-validada
grid_xgb = GridSearchCV(xgb, param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid_xgb.fit(X_train_sm, y_train_sm)

best_xgb = grid_xgb.best_estimator_
y_pred_xgb = best_xgb.predict(X_test)
y_proba_xgb = best_xgb.predict_proba(X_test)[:, 1]

print(f"Melhores parâmetros encontrados: {grid_xgb.best_params_}")


## 5. Avaliação do Modelo OtimizadoAvaliamos o poder do modelo utilizando a AUC-ROC (Métrica que leva em conta a separabilidade das classes) e a Acurácia Direta.

In [ ]:
print(f"Acurácia Global: {accuracy_score(y_test, y_pred_xgb):.4f}")
print(f"AUC-ROC (Poder de Separação): {roc_auc_score(y_test, y_proba_xgb):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))


In [ ]:
# Plotagem Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_proba_xgb)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='#8257E5', lw=2, label=f'XGBoost Otimizado (AUC={roc_auc_score(y_test, y_proba_xgb):.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.title('Curva ROC - Capacidade do Modelo em Detectar Risco')
plt.xlabel('Taxa Falsos Positivos')
plt.ylabel('Taxa Verdadeiros Positivos')
plt.legend()
plt.show()


### Feature Importance do Modelo OtimizadoQuais características o modelo julgou mais essenciais para decretar o risco?

In [ ]:
importances = pd.DataFrame({'Feature': cols_features, 'Importância': best_xgb.feature_importances_}).sort_values(by='Importância', ascending=False)
sns.barplot(data=importances, x='Importância', y='Feature', palette='magma')
plt.title('Features Mais Influentes (XGBoost Otimizado)')
plt.show()


## 6. Salvar o Modelo Final em PickleEste arquivo será exportado para ser lidado pelo nosso portal no Streamlit.

In [ ]:
modelo_final = {
    'modelo': best_xgb,
    'le_genero': le_genero,
    'le_bolsa': le_bolsa,
    'colunas': cols_features
}

with open('modelo_risco.pkl', 'wb') as f:
    pickle.dump(modelo_final, f)

print("Modelo 'modelo_risco.pkl' devidamente otimizado e exportado com sucesso!")
